In [1]:
import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (make_scorer, recall_score, f1_score, balanced_accuracy_score,
                             average_precision_score, confusion_matrix, classification_report)
from sklearn.inspection import permutation_importance

SEED = 42
TARGET = "collision_severity"
CLASSES = [1, 2, 3]  # 1 = Fatal, 2 = Serious, 3 = Slight

out_path = Path("/content/Output")
out_path.mkdir(parents=True, exist_ok=True)

train = pd.read_csv("/content/train_raw_classification.csv", low_memory=False)
val = pd.read_csv("/content/validation_raw_classification.csv", low_memory=False)
test = pd.read_csv("/content/test_raw_classification.csv", low_memory=False)

assert train.columns.tolist() == val.columns.tolist() == test.columns.tolist()
for name, d in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:10s}: {d.shape}")

train     : (359660, 50)
validation: (77070, 50)
test      : (77071, 50)


In [2]:
num_cols = [
    "speed_limit", "number_of_vehicles", "latitude", "longitude",
    "min_driver_age", "max_driver_age", "avg_driver_age", "vehicle_type_diversity",
]
cat_cols = [
    "road_type", "first_road_class", "junction_detail", "junction_control",
    "pedestrian_crossing", "urban_or_rural_area", "trunk_road_flag",
    "light_conditions", "weather_conditions", "road_surface_conditions",
    "special_conditions_at_site", "carriageway_hazards",
    "day_of_week", "season", "time_of_day_bucket",
]
cyc_cols = ["month_sin", "month_cos", "hour_of_day_sin", "hour_of_day_cos"]
flag_cols = [c for c in train.columns if c.startswith("has_")] + ["is_weekend"]
FEATURES = num_cols + cat_cols + cyc_cols + flag_cols

missing = [c for c in FEATURES + [TARGET] if c not in train.columns]
assert not missing, f"Missing columns: {missing}"

X_tr, y_tr = train[FEATURES], train[TARGET]
X_va, y_va = val[FEATURES], val[TARGET]
X_te, y_te = test[FEATURES], test[TARGET]
print("Features:", len(FEATURES))

Features: 36


In [3]:
def make_preprocessor():
    return ColumnTransformer(
        [
            ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
            ("cat", make_pipeline(
                SimpleImputer(strategy="most_frequent"),
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ), cat_cols),
            ("pass", "passthrough", cyc_cols + flag_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def make_svm(C=1.0, gamma="scale"):
    return Pipeline([
        ("prep", make_preprocessor()),
        ("clf", SVC(kernel="rbf", C=C, gamma=gamma, class_weight="balanced",
                    cache_size=1000, random_state=SEED)),
    ])


def make_linear_control(C=1.0):
    return Pipeline([
        ("prep", make_preprocessor()),
        ("clf", LinearSVC(C=C, class_weight="balanced", max_iter=5000, random_state=SEED)),
    ])

In [4]:
TUNE_N = 30000
X_tune, _, y_tune, _ = train_test_split(
    X_tr, y_tr, train_size=TUNE_N, stratify=y_tr, random_state=SEED
)
print("Tuning subsample:", X_tune.shape)
print("Class % in subsample:", (y_tune.value_counts(normalize=True) * 100).round(2).sort_index().to_dict())

Tuning subsample: (30000, 36)
Class % in subsample: {1: 1.47, 2: 22.74, 3: 75.79}


In [5]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
scoring = {
    "macro_f1": "f1_macro",
    "fatal_recall": make_scorer(recall_score, labels=[1], average="macro", zero_division=0),
}

baseline_res = cross_validate(make_svm(C=1.0, gamma="scale"), X_tune, y_tune,
                              cv=cv, scoring=scoring, n_jobs=-1)
print(f"Baseline (C=1, rbf, gamma=scale), 10-fold macro F1: "
      f"{baseline_res['test_macro_f1'].mean():.4f} ± {baseline_res['test_macro_f1'].std():.4f}")
print(f"Baseline 10-fold Fatal recall:  "
      f"{baseline_res['test_fatal_recall'].mean():.4f} ± {baseline_res['test_fatal_recall'].std():.4f}")

Baseline (C=1, rbf, gamma=scale), 10-fold macro F1: 0.3896 ± 0.0099
Baseline 10-fold Fatal recall:  0.2130 ± 0.0639


In [7]:
param_grid = {"clf__C": [0.1, 1, 10], "clf__gamma": ["scale", 0.01, 0.1]}
search = GridSearchCV(
    make_svm(), param_grid, scoring="f1_macro", cv=cv,
    n_jobs=-1, refit=True,
)
search.fit(X_tune, y_tune)

cv_table = pd.DataFrame(search.cv_results_)[
    ["param_clf__C", "param_clf__gamma", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score")
cv_table.to_csv(out_path / "svm_gridsearch_cv.csv", index=False)
print(cv_table.to_string(index=False))

best_C = search.best_params_["clf__C"]
best_gamma = search.best_params_["clf__gamma"]
print(f"Best: C={best_C}, gamma={best_gamma}, 10-fold macro F1={search.best_score_:.4f}")

 param_clf__C param_clf__gamma  mean_test_score  std_test_score  rank_test_score
          1.0              0.1         0.394487        0.010529                1
         10.0            scale         0.392632        0.011857                2
          1.0            scale         0.389553        0.009892                3
         10.0              0.1         0.379992        0.006262                4
         10.0             0.01         0.377151        0.008683                5
          0.1              0.1         0.372876        0.006737                6
          0.1            scale         0.362079        0.005989                7
          1.0             0.01         0.361906        0.005216                8
          0.1             0.01         0.351547        0.005317                9
Best: C=1, gamma=0.1, 10-fold macro F1=0.3945


In [8]:
control = make_linear_control(C=1.0)
control.fit(X_tr, y_tr)
print("Linear control fitted on the full training split.")

Linear control fitted on the full training split.


In [10]:
def evaluate(scores, y_true, bias=0.0):
    adjusted = scores + np.array([bias, 0.0, 0.0])  # bias is added to the Fatal column
    pred = np.array(CLASSES)[np.argmax(adjusted, axis=1)]
    recalls = recall_score(y_true, pred, labels=CLASSES, average=None, zero_division=0)
    Y = label_binarize(y_true, classes=CLASSES)
    metrics = {
        "macro_f1": f1_score(y_true, pred, average="macro"),
        "fatal_recall": recalls[0],
        "serious_recall": recalls[1],
        "slight_recall": recalls[2],
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "macro_avg_precision": average_precision_score(Y, scores, average="macro"),
        "accuracy_context_only": (pred == np.asarray(y_true)).mean(),
    }
    return metrics, pred


best = search.best_estimator_
assert list(best.classes_) == CLASSES, f"Unexpected class order: {best.classes_}"

scores_va_svm = best.decision_function(X_va)
scores_va_ctrl = control.decision_function(X_va)

m_svm, pred_svm = evaluate(scores_va_svm, y_va)
m_ctrl, _ = evaluate(scores_va_ctrl, y_va)
comparison = pd.DataFrame({"SVM (RBF, tuned)": m_svm, "Linear SVM (control)": m_ctrl}).round(4)
print(comparison.to_string())

print("Validation confusion matrix, SVM (rows = true 1,2,3; columns = predicted 1,2,3):")
print(confusion_matrix(y_va, pred_svm, labels=CLASSES))

                       SVM (RBF, tuned)  Linear SVM (control)
macro_f1                         0.4005                0.3899
fatal_recall                     0.0432                0.3239
serious_recall                   0.5644                0.1390
slight_recall                    0.6561                0.8980
balanced_accuracy                0.4212                0.4536
macro_avg_precision              0.4064                0.4259
accuracy_context_only            0.6262                0.7170
Validation confusion matrix, SVM (rows = true 1,2,3; columns = predicted 1,2,3):
[[   49   764   320]
 [  240  9890  7392]
 [  364 19727 38324]]


In [11]:
FATAL_RECALL_TARGET = 0.60  # PLACEHOLDER: agree the target with stakeholders before the final run

bias_grid = np.round(np.arange(0.0, 2.01, 0.05), 2)
bias_rows = []
for b in bias_grid:
    m, _ = evaluate(scores_va_svm, y_va, bias=b)
    bias_rows.append({"fatal_bias": b, **m})
bias_table = pd.DataFrame(bias_rows)

feasible = bias_table[bias_table["fatal_recall"] >= FATAL_RECALL_TARGET]
if len(feasible):
    chosen = feasible.sort_values("macro_f1", ascending=False).iloc[0]
else:
    chosen = bias_table.sort_values("fatal_recall", ascending=False).iloc[0]
    print("WARNING: target Fatal recall not reached on validation; using the highest Fatal recall found.")

FATAL_BIAS = float(chosen["fatal_bias"])
bias_table.to_csv(out_path / "svm_fatal_bias_validation.csv", index=False)
print(f"Chosen Fatal bias: {FATAL_BIAS}")
print(chosen[["fatal_bias", "macro_f1", "fatal_recall", "serious_recall", "slight_recall"]].to_string())

Chosen Fatal bias: 1.45
fatal_bias        1.450000
macro_f1          0.407587
fatal_recall      0.106796
serious_recall    0.546171
slight_recall     0.652281


In [12]:
rng = np.random.RandomState(SEED)
idx = rng.choice(len(X_va), size=5000, replace=False)
pi = permutation_importance(
    best, X_va.iloc[idx], y_va.iloc[idx],
    scoring="f1_macro", n_repeats=5, random_state=SEED, n_jobs=-1,
)
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": pi.importances_mean,
    "importance_std": pi.importances_std,
}).sort_values("importance_mean", ascending=False)
importance.to_csv(out_path / "svm_permutation_importance_validation.csv", index=False)
print(importance.head(20).to_string(index=False))

               feature  importance_mean  importance_std
    number_of_vehicles         0.009249        0.003075
               has_car         0.007469        0.003862
               has_hgv         0.006426        0.002052
       hour_of_day_cos         0.006358        0.004933
   urban_or_rural_area         0.006105        0.005697
vehicle_type_diversity         0.005140        0.003761
           speed_limit         0.004950        0.005852
    weather_conditions         0.004397        0.004325
                season         0.004210        0.003041
has_van_or_goods_light         0.004188        0.000961
       has_pedal_cycle         0.003667        0.003179
        min_driver_age         0.003278        0.005578
       junction_detail         0.002396        0.005460
        has_motorcycle         0.002276        0.002505
             road_type         0.001689        0.004824
    time_of_day_bucket         0.001563        0.005370
      light_conditions         0.001445        0

In [13]:
FLAG = out_path / "test_scored.flag"
assert not FLAG.exists(), "The test split has already been scored. Delete the flag file only for a deliberate re-run."

X_trval = pd.concat([X_tr, X_va])
y_trval = pd.concat([y_tr, y_va])
X_final, _, y_final, _ = train_test_split(
    X_trval, y_trval, train_size=TUNE_N, stratify=y_trval, random_state=SEED
)

final = make_svm(C=best_C, gamma=best_gamma)
final.fit(X_final, y_final)

scores_te = final.decision_function(X_te)
m_test, pred_te = evaluate(scores_te, y_te, bias=FATAL_BIAS)
FLAG.write_text("scored once")

print("Test metrics (scored once):")
print(pd.Series(m_test).round(4).to_string())
print(classification_report(y_te, pred_te, labels=CLASSES,
                            target_names=["Fatal", "Serious", "Slight"], digits=4))
print("Test confusion matrix (rows = true 1,2,3; columns = predicted 1,2,3):")
print(confusion_matrix(y_te, pred_te, labels=CLASSES))

Test metrics (scored once):
macro_f1                 0.4113
fatal_recall             0.1209
serious_recall           0.5512
slight_recall            0.6496
balanced_accuracy        0.4406
macro_avg_precision      0.4069
accuracy_context_only    0.6195
              precision    recall  f1-score   support

       Fatal     0.0793    0.1209    0.0958      1133
     Serious     0.3224    0.5512    0.4069     17522
      Slight     0.8360    0.6496    0.7311     58416

    accuracy                         0.6195     77071
   macro avg     0.4126    0.4406    0.4113     77071
weighted avg     0.7082    0.6195    0.6481     77071

Test confusion matrix (rows = true 1,2,3; columns = predicted 1,2,3):
[[  137   699   297]
 [  719  9658  7145]
 [  871 19596 37949]]


In [14]:
joblib.dump(final, out_path / "svm_final_pipeline.joblib")

summary = {
    "seed": SEED,
    "tuning_subsample_rows": TUNE_N,
    "final_training_rows": TUNE_N,
    "best_params": {"C": float(best_C), "gamma": str(best_gamma)},
    "baseline_10fold_macro_f1_mean": float(baseline_res["test_macro_f1"].mean()),
    "baseline_10fold_macro_f1_std": float(baseline_res["test_macro_f1"].std()),
    "gridsearch_10fold_macro_f1": float(search.best_score_),
    "fatal_recall_target": FATAL_RECALL_TARGET,
    "fatal_bias": FATAL_BIAS,
    "validation": {k: float(v) for k, v in m_svm.items()},
    "validation_linear_control": {k: float(v) for k, v in m_ctrl.items()},
    "test": {k: float(v) for k, v in m_test.items()},
    "note": "RBF SVM is tuned and finally trained on a stratified subsample; the linear control uses the full training split.",
}
with open(out_path / "svm_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved to", out_path)
for p in sorted(out_path.iterdir()):
    print(" -", p.name)

Saved to /content/Output
 - svm_fatal_bias_validation.csv
 - svm_final_pipeline.joblib
 - svm_gridsearch_cv.csv
 - svm_permutation_importance_validation.csv
 - svm_summary.json
 - test_scored.flag


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
